## PortAventura Park - Interactive Attraction Map

The amusement park in this dataset is **PortAventura World**, located in Salou/Vila-seca, Catalonia, Spain (41.087°N, 1.157°E).

Since the dataset uses anonymized attraction names, we:
1. Fetch real attraction coordinates from **OpenStreetMap** (Overpass API)
2. Build a **speculative mapping** between anonymized dataset names and real PortAventura rides (based on ride type/characteristics)
3. Render an **interactive map** with color-coded markers using Folium

In [ ]:
%pip install folium requests -q

import folium
import requests
import json

# =============================================================================
# Step 1: Fetch real PortAventura attraction coordinates from OpenStreetMap
# =============================================================================

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
BBOX = "(41.075,1.145,41.100,1.170)"  # Bounding box around PortAventura

def query_overpass(query_str):
    """Query the Overpass API and return named attraction dict {name: (lat, lon, type)}."""
    try:
        resp = requests.get(OVERPASS_URL, params={"data": query_str}, timeout=30)
        data = resp.json()
        attractions = {}
        for el in data.get('elements', []):
            name = el.get('tags', {}).get('name')
            if not name:
                continue
            if 'center' in el:
                lat, lon = el['center']['lat'], el['center']['lon']
            elif 'lat' in el:
                lat, lon = el['lat'], el['lon']
            else:
                continue
            if name not in attractions:
                atype = el.get('tags', {}).get('attraction', '')
                attractions[name] = (lat, lon, atype)
        return attractions
    except Exception as e:
        print(f"Overpass query failed: {e}")
        return {}

# Query 1: All tagged attractions/rides in the PortAventura area
q1 = f"""
[out:json][timeout:30];
(
  nwr["attraction"]{BBOX};
  nwr["tourism"="attraction"]{BBOX};
  nwr["leisure"="amusement_ride"]{BBOX};
);
out center;
"""

# Query 2: Specific rides by name that might only appear as buildings/areas
q2 = f"""
[out:json][timeout:30];
(
  nwr["name"~"Dragon|Stampida|Grand Canyon|Tutuki|Kontiki|Street Mission|Templo|Ferrocarril|La Granja|Magic Fish|Cobra|Waikiki|Canoe",i]{BBOX};
);
out center;
"""

print("Querying OpenStreetMap Overpass API...")
osm_data = query_overpass(q1)
osm_data2 = query_overpass(q2)

# Merge (first query takes priority for duplicates)
for name, val in osm_data2.items():
    if name not in osm_data:
        osm_data[name] = val

print(f"Found {len(osm_data)} unique named attractions from OSM")

# Filter to only PortAventura Park attractions (exclude Ferrari Land, water park, etc.)
FERRARI_LAND_NAMES = {
    'Junior Red Force', 'Racing Legends', 'Flying Dreams', 'Kids Podium',
    'Pole Position Challenge', 'Junior Championship', 'Red Force', 'Kids Tower',
    'Champions Race', 'Crazy Pistons', 'Thrill Towers', 'Maranello Grand Race', 'Colosseo'
}

portaventura_osm = {
    name: (lat, lon, atype)
    for name, (lat, lon, atype) in osm_data.items()
    if name not in FERRARI_LAND_NAMES
    and not name.startswith('Photo ')
    and 'Torre d' not in name
}

print(f"After filtering (PortAventura Park only): {len(portaventura_osm)} attractions")
for name, (lat, lon, atype) in sorted(portaventura_osm.items()):
    print(f"  {name}: ({lat:.6f}, {lon:.6f})  [{atype}]")

In [ ]:
# =============================================================================
# Step 2: Speculative mapping – Anonymized dataset names → Real PortAventura rides
# =============================================================================
# NOTE: This mapping is SPECULATIVE. The dataset uses generic anonymized names.
# We match based on ride type/characteristics. Some mappings are confident,
# others are best-guesses.

ANONYMIZED_TO_REAL = {
    # --- High-confidence mappings (ride type is distinctive) ---
    'Giga Coaster':      'Shambhala',              # Tallest coaster, hyper/giga class (76m)
    'Roller Coaster':    'Dragon Khan',             # Classic main steel coaster, 8 inversions
    'Flying Coaster':    'Furius Baco',             # Wing/flying seating, launched coaster
    'Kiddie Coaster':    'Tami Tami',               # Children's family coaster (SésamoAventura)
    'Drop Tower':        'Hurakan Condor',          # 100m free-fall drop tower (México)
    'Rapids Ride':       'Grand Canyon Rapids',     # Whitewater raft ride (Far West)
    'Water Ride':        'Tutuki Splash',           # Volcano-themed water flume (Polynesia)
    'Bumper Cars':       'Crazy Barrels',           # Bumper/spinning ride (Far West)
    'Merry Go Round':    'Carousel',                # Classic carousel (Far West)

    # --- Medium-confidence mappings ---
    'Spinning Coaster':  'El Diablo - Tren de la Mina',  # Mine train coaster (México)
    'Inverted Coaster':  'Uncharted',               # Modern coaster with drops/inversions (Far West)
    'Circus Train':      'Coco Pilot',              # Train ride around the park area
    'Haunted House':     'Street Mission',           # Interactive dark ride (SésamoAventura)
    'Go-Karts':          'Cobra Imperial',           # Tracked ride in China area
    'Swing Ride':        'VolPaiute',               # Swing/spinning flat ride (Far West)
    'Vertical Drop':     'Stampida',                # Wooden dueling coaster with steep drops (Far West)
    'Crazy Dance':       'Serpiente Emplumada',     # Spinning ride (México)
    'Oz Theatre':        'Templo del Fuego',        # Major show/experience venue (México)
    'Superman Ride':     'Tomahawk',                # Coaster (Far West, junior wooden coaster)

    # --- Lower-confidence / best-guess mappings ---
    'Himalaya Ride':     'Wild Buffalos',           # Ride in Far West area
    'Giant Wheel':       'Angkor',                  # Large attraction in China area
    'Spiral Slide':      'Silver River Flume',      # Water flume slide (Far West)
    'Free Fall':         'El Salto de Blas',        # Drop tower (SésamoAventura, kids)
    'Dizzy Dropper':     'Buffalo Rodeo',           # Spinning ride (Far West)
    'Bungee Jump':       'Hysteria in Boothill',    # Extreme experience (Far West)
    'Zipline':           'Naikiki',                 # Adventure ride (SésamoAventura area)
}

# Park area classification for color coding
AREA_COLORS = {
    'Mediterrània': 'blue',
    'China':        'red',
    'México':       'orange',
    'Far West':     'beige',
    'Polynesia':    'green',
    'SésamoAventura': 'purple',
}

REAL_NAME_TO_AREA = {
    'Furius Baco': 'Mediterrània',
    'Dragon Khan': 'China', 'Shambhala': 'China', 'Angkor': 'China',
    'Cobra Imperial': 'China',
    'Hurakan Condor': 'México', 'El Diablo - Tren de la Mina': 'México',
    'Serpiente Emplumada': 'México', 'Templo del Fuego': 'México',
    'Grand Canyon Rapids': 'Far West', 'Silver River Flume': 'Far West',
    'Stampida': 'Far West', 'Tomahawk': 'Far West', 'Carousel': 'Far West',
    'VolPaiute': 'Far West', 'Vultrix': 'Far West', 'Wild Buffalos': 'Far West',
    'Buffalo Rodeo': 'Far West', 'Crazy Barrels': 'Far West',
    'Uncharted': 'Far West', 'Hysteria in Boothill': 'Far West',
    'Tutuki Splash': 'Polynesia', 'Canoes': 'Polynesia',
    'Tami Tami': 'SésamoAventura', 'Coco Pilot': 'SésamoAventura',
    'El Salto de Blas': 'SésamoAventura', 'Street Mission': 'SésamoAventura',
    'La Granja De Elmo': 'SésamoAventura', 'Magic Fish': 'SésamoAventura',
    'Waikiki': 'SésamoAventura', 'Naikiki': 'SésamoAventura',
    'Kiddie Dragons': 'SésamoAventura',
}

# Build coordinate lookup: real_name -> (lat, lon)
# Start with comprehensive hardcoded coordinates from previous OSM queries + RCDB
# This ensures the map works even if the Overpass API is unavailable/rate-limited
FALLBACK_COORDS = {
    # --- From OSM Overpass API (verified coordinates) ---
    'Dragon Khan':               (41.0884741, 1.1608395),   # China - steel coaster, 8 inversions
    'Shambhala':                 (41.0884592, 1.1619114),   # China - hyper coaster, tallest in park
    'Furius Baco':               (41.0843941, 1.1563790),   # Mediterrània - launched wing coaster
    'Stampida':                  (41.0907598, 1.1560799),   # Far West - wooden dueling coaster
    'Tomahawk':                  (41.0899839, 1.1570706),   # Far West - junior wooden coaster
    'El Diablo - Tren de la Mina': (41.0890787, 1.1581803), # México - mine train coaster
    'Tami Tami':                 (41.0866994, 1.1593646),   # SésamoAventura - kids coaster
    'Hurakan Condor':            (41.0899185, 1.1590092),   # México - 100m drop tower
    'Grand Canyon Rapids':       (41.0868132, 1.1565065),   # Far West - raft ride
    'Silver River Flume':        (41.0879321, 1.1562076),   # Far West - log flume
    'Tutuki Splash':             (41.0846798, 1.1580606),   # Polynesia - water ride
    'Angkor':                    (41.0868121, 1.1622340),   # China - splash battle
    'Cobra Imperial':            (41.0881663, 1.1584035),   # China - tracked ride
    'Uncharted':                 (41.0896950, 1.1555904),   # Far West - modern coaster
    'Street Mission':            (41.0863630, 1.1605387),   # SésamoAventura - dark ride
    'Templo del Fuego':          (41.0904439, 1.1597998),   # México - fire show
    'Serpiente Emplumada':       (41.0900656, 1.1587030),   # México - spinning ride
    'Carousel':                  (41.0896774, 1.1558920),   # Far West - carousel
    'VolPaiute':                 (41.0895807, 1.1574080),   # Far West - swing ride
    'Vultrix':                   (41.0897599, 1.1570119),   # Far West - ride
    'Wild Buffalos':             (41.0873430, 1.1573234),   # Far West - ride
    'Buffalo Rodeo':             (41.0875486, 1.1569792),   # Far West - ride
    'Crazy Barrels':             (41.0870950, 1.1570012),   # Far West - spinning ride
    'Coco Pilot':                (41.0862885, 1.1592522),   # SésamoAventura - monorail
    'El Salto de Blas':          (41.0865605, 1.1592712),   # SésamoAventura - kids drop tower
    'La Granja De Elmo':         (41.0866087, 1.1590028),   # SésamoAventura - farm ride
    'Magic Fish':                (41.0866703, 1.1599224),   # SésamoAventura - water carousel
    'Waikiki':                   (41.0866852, 1.1587124),   # SésamoAventura - ride
    'Naikiki':                   (41.0869735, 1.1600437),   # SésamoAventura - ride
    'Kiddie Dragons':            (41.0869326, 1.1601842),   # SésamoAventura - ride
    'Canoes':                    (41.0862036, 1.1584369),   # Polynesia - junior water ride
    'Hysteria in Boothill':      (41.0902704, 1.1567159),   # Far West - experience
    # --- Estimated coordinates (not found in OSM, placed by park area) ---
    'Kontiki':                   (41.0852000, 1.1583000),   # Polynesia - pirate ship
    'Dodgems':                   (41.0892000, 1.1558000),   # Far West - bumper cars
    'Yucatán':                   (41.0897000, 1.1592000),   # México - spinning ride
    'Armadillos':                (41.0898000, 1.1595000),   # México - kids ride
    'Tea Cups':                  (41.0878000, 1.1605000),   # China - tea cups
    'Driving School':            (41.0876000, 1.1608000),   # China - go-karts
    'Los Potrillos':             (41.0901000, 1.1594000),   # México - kids ride
}

# Use live OSM data when available, fall back to hardcoded coords
REAL_COORDS = dict(FALLBACK_COORDS)  # start with all fallbacks
for name, (lat, lon, _) in portaventura_osm.items():
    if name == 'STAMPIDA':
        REAL_COORDS['Stampida'] = (lat, lon)
    elif name.startswith('Stampida -'):
        continue  # skip individual tracks
    else:
        REAL_COORDS[name] = (lat, lon)  # OSM overrides fallback

print(f"Total real attraction coordinates: {len(REAL_COORDS)}")
print(f"Anonymized-to-real mappings: {len(ANONYMIZED_TO_REAL)}")

# Verify all mapped real names have coordinates
missing = [real for real in ANONYMIZED_TO_REAL.values() if real not in REAL_COORDS]
if missing:
    print(f"WARNING - Missing coordinates for: {missing}")
else:
    print("All mapped attractions have coordinates.")

In [ ]:
# =============================================================================
# Step 3: Build interactive Folium map
# =============================================================================

# Create base map centered on PortAventura Park
m = folium.Map(
    location=[41.087, 1.157],
    zoom_start=16,
    tiles='OpenStreetMap',
    max_zoom=19,
)

# --- Add mapped dataset attractions (main markers) ---
for anon_name, real_name in ANONYMIZED_TO_REAL.items():
    if real_name not in REAL_COORDS:
        continue
    lat, lon = REAL_COORDS[real_name]
    area = REAL_NAME_TO_AREA.get(real_name, 'Unknown')
    color = AREA_COLORS.get(area, 'gray')

    # Build popup HTML – show only the dataset (fictional) name and area
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; min-width: 180px;">
        <b style="font-size: 14px;">{anon_name}</b><br>
        <span style="color: #444; font-size: 12px;">Area: {area}</span>
    </div>
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=anon_name,
        icon=folium.Icon(color=color, icon='star', prefix='fa'),
    ).add_to(m)

# --- Add legend ---
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 12px 16px; border-radius: 8px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-family: Arial, sans-serif;
            font-size: 12px; line-height: 1.6;">
    <b style="font-size: 13px;">Park Areas</b><br>
    <span style="color: #3388ff;">&#9733;</span> Mediterrània<br>
    <span style="color: #dc3545;">&#9733;</span> China<br>
    <span style="color: #fd7e14;">&#9733;</span> México<br>
    <span style="color: #d2b48c;">&#9733;</span> Far West<br>
    <span style="color: #28a745;">&#9733;</span> Polynesia<br>
    <span style="color: #6f42c1;">&#9733;</span> SésamoAventura
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
print(f"Map shows {len(ANONYMIZED_TO_REAL)} dataset attractions")
m

In [ ]:
# =============================================================================
# Step 4: Save the map as standalone HTML & display the mapping table
# =============================================================================

# Save interactive map to HTML
map_path = '../portaventura_attractions_map.html'
m.save(map_path)
print(f"Interactive map saved to: {map_path}")

# Display the mapping as a clean DataFrame
mapping_rows = []
for anon_name, real_name in sorted(ANONYMIZED_TO_REAL.items()):
    area = REAL_NAME_TO_AREA.get(real_name, 'Unknown')
    if real_name in REAL_COORDS:
        lat, lon = REAL_COORDS[real_name]
        coord_str = f"({lat:.5f}, {lon:.5f})"
        source = "OSM" if real_name in portaventura_osm or real_name == 'Stampida' else "OSM (cached)"
    else:
        coord_str = "N/A"
        source = "N/A"
    mapping_rows.append({
        'Dataset Name': anon_name,
        'Real Name (speculative)': real_name,
        'Area': area,
        'Coordinates': coord_str,
        'Source': source,
    })

mapping_df = pd.DataFrame(mapping_rows)
print(f"\n{'='*80}")
print("SPECULATIVE MAPPING: Anonymized Dataset Names → Real PortAventura Attractions")
print(f"{'='*80}")
mapping_df